# 03_00 Attention (overview)

*📖 Book: chapter 3*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → [model reads] → model learns → model writes → MCP tool
> ```
>
> You are here: **model reads**
>
> Project files this connects to:
>
> - `src/homer_ai/model.py`
>
> **This notebook:** To guess the next letter well, the model must look back at the letters before it, and know which ones matter. That is attention, the heart of every modern language model.
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- Why guessing the next letter needs more than the last letter
- What attention is: deciding which earlier letters matter, and by how much
- The three ideas of the chapter: scores and mixing, no peeking, many heads
- Which part of our `model.py` is attention
- Which notebook to open for each idea

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **attention** | the model's way of giving each earlier letter a share of its focus, then mixing them by those shares | while guessing after `μοῦσ`, `σ` gets the biggest share |
| **context** | the earlier letters the model can look at when it makes one guess | for the guess after `μοῦσ`, the context is `μ ο ῦ σ` |

## 3. The idea

### ❓ Why isn't the last letter enough?

Picture the letters of a line as people who walk into a room one by one and sit down round a table. Each of them has one job, to guess who walks in next, and knowing only itself it would be guessing blind. So it looks round the table and decides whom to listen to: a lot to the ones beside it, a little to someone further away, and nothing to anyone who hasn't arrived yet. What it takes away is one mix of their voices, and it guesses from that. The picture is too kind in one way: people choose whom to listen to by thinking, and a letter does it with a multiplication that only gets good through training.

Now the same thing on real text. Read the first line of the *Odyssey*: `ἄνδρα μοι ἔννεπε, μοῦσα` ("tell me, Muse, of the man"). Say you've read `μοῦσ` and have to guess the next letter. If all you saw was the last letter, `σ`, you could be at the end of hundreds of different words. The letters just before it (`ο ῦ`) narrow it down a lot.

```
μ  ο  ῦ  σ  →  ?
      └──┴──┘ the letters just before matter most
```

We can actually count this in the real Book 1. Below, we look at every place where the text has the letters `before`, and see which letter comes next.

👉 **Predict first:** with more letters in front, will the most common next letter be more certain, or less?

In [ ]:
from collections import Counter

from homer_ai.train_homer import BOOKS_DIR, load_books

text = load_books(BOOKS_DIR)[0]

for before in ["σ", "οῦσ", "μοῦσ"]:
    nexts = Counter()
    for i in range(len(text) - len(before)):
        # does the text, from place i, begin with `before`?
        if text.startswith(before, i):
            nexts[text[i + len(before)]] += 1
    total = sum(nexts.values())
    best, n = nexts.most_common(1)[0]
    print(f"after {before!r:8} seen {total:>3} times;", end=" ")
    print(f"most common next {best!r}: {n} of {total} = {n / total:.0%}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

After `σ` alone, the best guess is right only about a quarter of the time. After `οῦσ` it's about half. After `μοῦσ` it's always `α` (though that only happened once, so don't read much into that one; it's the direction that counts). By the way, `Counter` is a ready-made Python tool that counts things, and `text.startswith(before, i)` asks "from place `i`, does the text begin with `before`?". So more earlier letters make the guess more certain. The model has to **look back**.

</details>

### ❓ Do all the earlier letters matter equally?

No. The letters right before matter most, and a letter far back matters less. But a few far-back ones can still matter a lot (the first letter of a word can tell you the ending, for example). So the model also has to decide which earlier letters to listen to, and how much.

That's **attention**. Think of 100 points of focus, shared out between the earlier letters. Here's an invented picture of the idea (the real numbers come in [03_01](03_01_simple_attention.ipynb)):

```
guessing after μοῦσ:   μ ▏        a little
                       ο ▏▏▏▏
                       ῦ ▏▏▏▏▏▏
                       σ ▏▏▏▏▏▏▏▏▏▏  the most
```

Then the model mixes the earlier letters, each one counting as much as its share. What comes out is one list of numbers that sums up the **context**: what the model needs from everything it has read so far.

### ❓ Where is attention in our `model.py`?

Right in the middle of `forward`, between the embedding (chapter 2) and the final `head` that makes the 137 scores. Let's print the real lines of `forward`, so you can see where it sits.

👉 **Predict first:** which lines do you think are attention?

In [ ]:
import inspect

from homer_ai.model import HomerGreekLLM

print(inspect.getsource(HomerGreekLLM.forward))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`x = self.embedding(idx)` is the numbers from chapter 2. Everything from `Q, K, V = …` down to `context = weights @ V` is **attention**. Then `self.head(context)` turns the mixed result into 137 scores, one per token. (Chapter 4 builds the rest of the model around this.)

</details>

### ❓ What are the three ideas in this chapter?

1. **Score and mix.** How much does each earlier letter matter (a score), and how do we mix them by those scores?
2. **No peeking.** While it guesses a letter, the model mustn't see the letters that come after it, or it would just copy the answer.
3. **Many heads.** One way of looking may not be enough, so we let several small "lookers" each search for something different.

### ❓ Which notebook do I open for each idea?

| Notebook | Question it answers |
|---|---|
| [`03_01_simple_attention.ipynb`](03_01_simple_attention.ipynb) | How do we score the earlier letters and mix them, with nothing to learn yet? |
| [`03_02_trainable_attention.ipynb`](03_02_trainable_attention.ipynb) | How does the model learn what to look for? (query, key, value) |
| [`03_03_hiding_the_future.ipynb`](03_03_hiding_the_future.ipynb) | How do we stop the model peeking at later letters? |
| [`03_04_many_heads.ipynb`](03_04_many_heads.ipynb) | What if several heads look at once? |

### ❓ Is everything in this chapter in our `model.py`?

Most of it. Our model has **one** attention head and the mask that stops peeking (ideas 1 and 2). It has no second head (idea 3) and no positions ([02_06](../02_text_to_numbers/02_06_positions.ipynb)). The GPT of chapter 4 (`src/homer_ai/gpt.py`) has both: several heads ([04_04](../04_building_the_model/04_04_one_block.ipynb)) and positions ([04_05](../04_building_the_model/04_05_the_full_model.ipynb)).

## 4. Recap

- To guess the next letter well, the model must look back: more earlier letters make the guess more certain.
- Attention gives each earlier letter a share of focus (the shares add up to 1), then mixes the letters by those shares.
- Idea 1 is scores and mixing, idea 2 is not peeking ahead, idea 3 is several heads.

**Next:** [03_01: Simple attention](03_01_simple_attention.ipynb): scoring and mixing with nothing to learn.